# Exercise 4 - Spam Filtering with Naive Bayes

In this Exercise we will implement a spam detector that relies on the naive Bayes assumption. We will then compare its performance to logistic regression.

In the event of a persistent problem, do not hesitate to contact the course instructors under

- maurice.wenig@uni-jena.de

### Submission
- Deadline of submission: 25.05.2026 23:59
- Submission on [moodle page](https://moodle.uni-jena.de/course/view.php?id=76636)


# Dataset

We will use the [SMS Spam Collection Dataset](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset?resource=download). You find this dataset as `spam_data.csv`. Each line consists of a message together with a label:
- spam (message is a spam message)
- ham (message is legitimate)

### Task 1
Find a way to load the dataset and transform the features `X` (SMS) and the labels `Y` (spam/ham) into numerical representations.

For transforming SMS into features, check out the bag of words representation from [scikit-learn](https://scikit-learn.org/stable/modules/feature_extraction.html)

In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# Load data - grabbing only the first two columns to ignore Kaggle's empty unnamed columns
df = pd.read_csv('spam_data.csv', encoding='latin-1', usecols=[0, 1])
df.columns = ['label', 'message']

# Transform labels into boolean array
y = (df['label'] == 'spam').to_numpy()

# Transform SMS into numerical features
vectorizer = CountVectorizer()
x = vectorizer.fit_transform(df['message']).toarray()
print(np.sum(x))

# assertions
assert x.shape == (5573, 8798)
assert y.shape == (5573,)
assert np.sum(x) == 80997

81069


AssertionError: 

# Naive Bayes

The naive Bayes filter is based on the Bayes formula with an additional simplifying (naive) assumption about the nature of the likelihood.

Let $S = \{\text{spam}, \text{ham}\}$ be the source of a SMS and $W = [w_1,\dots w_k]$ be the sequence of words contained in the SMS.
Then filtering for spam and ham is done by evaluating the posterior distribution

\begin{align}
p(S|W)&=\cfrac{p(S)p(W|S)}{p(W)}
\end{align}

Lets look at the single parts of the equations right hand side and how to implement them.

As a running example we will use $W = [\text{this}, \text{is}, \text{no}, \text{spam}, \text{message}]$.

## Prior $p(S)$

The prior distribution $p(S)$ is independent of the message $W$. We will use the maximum likelihood (ML) estimate for a categorical distribution, which is the relative frequency of the categories among the dataset.

\begin{align}
p(S = \text{spam})_{ML} &= \cfrac{\text{\# SMS that are spam}}{\text{\# SMS}}\\
p(S = \text{ham})_{ML} &= 1 - p(S = \text{spam})
\end{align}

### Task 2

Estimate $p(S)$. Display the estimated distribution in a bar chart.

In [ ]:
import matplotlib.pyplot as plt

# Estimate p(S) as relative frequencies
p_spam = np.mean(y)
p_ham = 1 - p_spam

# Display in a bar chart
plt.bar(['Ham', 'Spam'], [p_ham, p_spam], color=['blue', 'red'])
plt.title('Prior Distribution p(S)')
plt.ylabel('Probability')
plt.show()

# assertions
assert np.isclose(p_spam, 0.13403911717207967)

## Likelihood p(W|S)

The likelihood distribution models how likely a SMS is, given we know its either spam or ham.

E.g. for $W = [\text{this}, \text{is}, \text{no}, \text{spam}, \text{message}]$ we would expect something like 

\begin{align}
p(W|S=\text{spam}) &= \text{low}\\
p(W|S=\text{ham}) &= \text{medium}\\
\end{align}

However to estimate $p(W|S)$ we would need a dataset with the exact same $W$ appearing in both contexts: spam and ham. Since this is not the case for our dataset, this is the part where we make a naive assumption:

\begin{align}
p(W|S) = \prod_{w\in W}p(w|S)
\end{align}

That is, we consider each word in the SMS text independend of the others. This simplification enables us to estimate the likelihood, since single words to in fact appear in both contexts.

For a single word $w$, we can again estimate the probability as relative frequency 

\begin{align}
p(w|S = \text{spam})_{ML} &= \cfrac{\text{\# word $w$ is labeled spam}}{\text{\# any word is labeled spam}}\\
p(w|S = \text{ham})_{ML} &= \cfrac{\text{\# word $w$ is labeled ham}}{\text{\# any word is labeled ham}}\\
\end{align}

Since we cannot expect every word to have appeared in a spam and ham message, we will smooth our dataset with a [Laplace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing) of $\varepsilon = 0.001$. 

This is done by adding $\varepsilon$ to the count of every word in every SMS.

As an example the count vector for a SMS over a vocabulary of 5 words is transformed from

\begin{align}
[1, 2, 0, 0, 1]
\end{align}

into

\begin{align}
[1.001, 2.001, 0.001, 0.001, 1.001]\,.
\end{align}

This way we do not have zero probabilities in the product for calculating $p(W|S)$.

### Task 3

Apply Laplace smoothing ($\varepsilon = 0.001$) to the dataset.

In [ ]:
# Apply Laplace smoothing with epsilon = 0.001
epsilon = 0.001
x_smooth = x + epsilon

# assertions
assert np.isclose(np.sum(x_smooth), 80997 + 5573 * 8798 * 0.001)

### Task 4

For $W = [\text{this}, \text{is}, \text{no}, \text{spam}, \text{message}]$, calculate $p(W|S)$.

Display $p(w|S = \text{spam}), p(w|S = \text{ham})$ for every $w\in W$.

In [ ]:
p_test_words_given_spam = [0.003713601454803985,0.006568721964475392,0.002886030292580366,7.228834102022863e-05,0.0014377807586891445]
p_test_words_given_ham = [0.002444957426685282,0.00702643050876389,0.002879488894263727,4.558801875075269e-05,0.0006312608663567718]

In [ ]:
sum(p_test_words_given_ham)

In [ ]:
np.isclose(sum(p_test_words_given_ham), 0.013027725714820422)

In [ ]:
# Calculate P(w|S) for the entire vocabulary first
spam_word_counts = np.sum(x_smooth[y], axis=0)
ham_word_counts = np.sum(x_smooth[~y], axis=0)

p_w_given_spam = spam_word_counts / np.sum(spam_word_counts)
p_w_given_ham = ham_word_counts / np.sum(ham_word_counts)

# Calculate for the specific test message
test_message = ["this", "is", "no", "spam", "message"]
word_indices = [vectorizer.vocabulary_[w] for w in test_message]

p_test_words_given_spam = p_w_given_spam[word_indices]
p_test_words_given_ham = p_w_given_ham[word_indices]

# assertions
assert np.isclose(sum(p_test_words_given_spam), 0.014678422811569116)
assert np.isclose(sum(p_test_words_given_ham), 0.013027725714820422)

# Plot p(w|S)
x_axis = np.arange(len(test_message))
width = 0.35

plt.bar(x_axis - width/2, p_test_words_given_ham, width, label='Ham')
plt.bar(x_axis + width/2, p_test_words_given_spam, width, label='Spam')
plt.xticks(x_axis, test_message)
plt.ylabel('Probability p(w|S)')
plt.title('Likelihood of individual words given label')
plt.legend()
plt.show()

### Task 5
From the dataset, list the top 5 words with the highest probabilities $p(w|S = \text{spam})$ and $p(w|S = \text{ham})$.

In [ ]:
# List top 5 words according to p(w|S)
feature_names = vectorizer.get_feature_names_out()

top_spam_indices = np.argsort(p_w_given_spam)[::-1][:5]
top_ham_indices = np.argsort(p_w_given_ham)[::-1][:5]

top_spam_words = [feature_names[i] for i in top_spam_indices]
top_ham_words = [feature_names[i] for i in top_ham_indices]

# assertions
assert set(top_spam_words) == {"to", "call", "you", "your", "free"}
assert set(top_ham_words) == {"you", "to", "the", "and", "in"}

## Evidence $p(W)$

The evidence tells us how likely the SMS was anyway. In many cases this is the most difficult part of the posterior to calculate. Here however we are lucky, since there are only two cases for $S$ and therefore

\begin{align}
p(W) = p(W|S=\text{spam}) + p(W|S=\text{ham})
\end{align}

That is $p(W)$ acts as a normalization constant.

### Task 6

For $W = [\text{this}, \text{is}, \text{no}, \text{spam}, \text{message}]$, calculate $p(W)$.

In [ ]:
# calculate P(W) for W = [this is no spam message]
# P(W) = P(W|Spam)*P(Spam) + P(W|Ham)*P(Ham)
likelihood_W_given_spam = np.prod(p_test_words_given_spam)
likelihood_W_given_ham = np.prod(p_test_words_given_ham)

p_test_message = (likelihood_W_given_spam * p_spam) + (likelihood_W_given_ham * p_ham)

assert np.isclose(p_test_message, 8.740660323205609e-15)

## Classification

With Prior, Likelihood and Evidence we can now assemble the Posterior

\begin{align}
p(S|W)&=\cfrac{p(S)p(W|S)}{p(W)}\,.
\end{align}

Remember that we want to use the Posterior to classify $W$:

\begin{align}
\kappa(W) = \begin{cases}
\text{spam}&\text{, if }p(S = \text{spam}|W) \geq p(S = \text{ham}|W)\\
\text{ham}&\text{, else}
\end{cases}
\end{align}

### Task 7

Implement the following `NaiveBayes` class. 

Use it to fit and predict on the dataset.

In [ ]:
import numpy.typing as npt
from typing import Self


class NaiveBayesClassifier():
    def __init__(self, laplace_smoothing_constant: float = 0.0001):
        """Class for binary naive Bayes."""

        self.laplace_regularization_constant = laplace_smoothing_constant
        # n_labels x n_words
        self.log_p_word_given_label: npt.NDArray[np.float64] = np.empty(0)
        # n_labels
        self.log_p_label: npt.NDArray[np.float64] = np.empty(0)

    def fit(self, x: npt.NDArray[np.float64], y: npt.NDArray[np.bool_]) -> Self:
        """Given a dataset of count vectors, calculates probabilities needed for prediction.

        Parameters
        ----------
        x : npt.NDArray[np.float64]
            Word count matrix (n_sms x n_words).
        y : npt.NDArray[np.bool_]
            Label matrix (n_sms).
        """

        # Calculate points
        p_spam = np.mean(y)
        p_ham = 1 - p_spam
        self.log_p_label = np.log(np.array([p_ham, p_spam]))
        
        # Apply smoothing
        x_smooth = x + self.laplace_regularization_constant
        
        # Calculate Likelihoods P(w|S)
        spam_counts = np.sum(x_smooth[y], axis=0)
        ham_counts = np.sum(x_smooth[~y], axis=0)
        
        p_w_spam = spam_counts / np.sum(spam_counts)
        p_w_ham = ham_counts / np.sum(ham_counts)
        
        # Store log probabilities (Shape: 2 x n_words)
        self.log_p_word_given_label = np.log(np.vstack((p_w_ham, p_w_spam)))
        
        return self


    def predict(self, x: npt.NDArray[np.float64]) -> npt.NDArray[np.bool_]:
        """Given a dataset of count vectors, predicts labels.

        Parameters
        ----------
        x : npt.NDArray[np.float64]
            Word count matrix (n_sms x n_words).

        Returns
        -------
        npt.NDArray[np.bool_]
            Vector of predictions for labels (0 = ham, 1 = spam).
        """
        # Evaluate proportional log posterior: log P(S) + sum(log P(w|S))
        # Matrix multiplication handles the sum of log likelihoods for the word counts
        log_posterior = self.log_p_label + x @ self.log_p_word_given_label.T
        
        # Predict 1 (Spam) if log_posterior for spam is higher, else 0 (Ham)
        return np.argmax(log_posterior, axis=1) == 1
    
    def accuracy(self, x: npt.NDArray[np.float64], y: npt.NDArray[np.bool_]) -> float:
        """Calculates accuracy for given dataset.

        Parameters
        ----------
        x : npt.NDArray[np.float64]
            Word count matrix (n_sms x n_words).
        y : npt.NDArray[np.bool_]
            Vector of true labels (0 = ham, 1 = spam).

        Returns
        -------
        float
            Percentage of correctly classified x.
        """
        predictions = self.predict(x)
        return np.mean(predictions == y)


# assertions
classifier = NaiveBayesClassifier(laplace_smoothing_constant=0.0001).fit(x, y)
assert classifier.accuracy(x, y) > 0.99

### Task 8

Obviously we trained on the same dataset as we tested and therefore cannot quite judge the performance of the naive Bayes classifier. 

Split your data into 75% training- and 25% testdata. Use a seed for reproducibility.

In [ ]:
from sklearn.model_selection import train_test_split

# Provide train + testsplit
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)

assert np.isclose(len(x_train) / len(x_test), 3, atol=0.01)

### Task 9

Now we systematically want to test our classifier. 

For different values of $\varepsilon$ track the accuracy on train and testdata.
Which value for $\varepsilon$ would you recommend?

In [ ]:
# Report train- and test accuracy for different epsilons
epsilons = [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0, 5.0, 10.0]

for eps in epsilons:
    clf = NaiveBayesClassifier(laplace_smoothing_constant=eps).fit(x_train, y_train)
    acc_train = clf.accuracy(x_train, y_train)
    acc_test = clf.accuracy(x_test, y_test)
    print(f"Epsilon: {eps:<8} | Train Acc: {acc_train:.4f} | Test Acc: {acc_test:.4f}")